<a href="https://colab.research.google.com/github/ce90045/llmfineTunningEdu/blob/main/deepseek_____aime_2024.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DeepSeek-R1-Distill-Llama-8B 모델 추론 능력 테스트하기(American Invitational Mathematics Examination (AIME) 데이터셋)
## 작성자 : AISchool ( http://aischool.ai/%ec%98%a8%eb%9d%bc%ec%9d%b8-%ea%b0%95%ec%9d%98-%ec%b9%b4%ed%85%8c%ea%b3%a0%eb%a6%ac/ )
## Reference : https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Llama-8B
## AIME 2024 데이터 : https://huggingface.co/datasets/Maxwell-Jia/AIME_2024?row=1

In [ ]:
!nvidia-smi

Mon Jan 27 08:27:18 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off | 00000000:00:04.0 Off |                    0 |
| N/A   33C    P0              46W / 400W |      2MiB / 40960MiB |      0%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

# 라이브러리 설치

In [ ]:
!pip install accelerate
!pip install --upgrade transformers
!pip install datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 101.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.47.1
    Uninstalling transformers-4.47.1:
      Successfully uninstalled transformers-4.47.1


# DeepSeek-R1-Distill-Llama-8B 모델 불러오기

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"

deepseek_tokenizer = AutoTokenizer.from_pretrained(model_id)
deepseek_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/3.06k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

model-00001-of-000002.safetensors:   0%|          | 0.00/8.67G [00:00<?, ?B/s]

model-00002-of-000002.safetensors:   0%|          | 0.00/7.39G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [ ]:
def generate_response(system_message, user_message, tokenizer, model):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]

    outputs = model.generate(
        input_ids,
        max_new_tokens=8192,
        eos_token_id=terminators,
        do_sample=False,
        temperature=0.6,
        top_p=0.95
    )
    response = outputs[0][input_ids.shape[-1]:]

    return tokenizer.decode(response, skip_special_tokens=True)

In [ ]:
print(deepseek_model.device)

cuda:0


In [ ]:
# 한국어 테스트
deepseek_inference_result = generate_response(system_message="",
                             user_message="안녕! 네 이름은 뭐야?",
                                            tokenizer=deepseek_tokenizer,
                                            model=deepseek_model)
print(deepseek_inference_result)

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


<think>
Alright, the user greeted me in Korean with "안녕! 네 이름은 뭐야?" which means "Hello! What's your name?"

I should respond in Korean to keep the conversation natural.

I need to introduce myself clearly.

I'll say my name is "DeepSeek-R1-Lite-Preview" and offer to help them with any questions they have.

Keeping it friendly and open-ended is important.
</think>

안녕! 내 이름은 "DeepSeek-R1-Lite-Preview"입니다. 어떻게 도와질 수 있을까요?


# AIME 2024 데이터셋에 대한 성능 테스트

## 1. DeepSeek-R1-Distill-Llama-8B 모델

In [ ]:
from datasets import load_dataset
aime_dataset = load_dataset("Maxwell-Jia/AIME_2024")
aime_dataset['train'][0]

{'ID': '2024-II-4',
 'Problem': 'Let $x,y$ and $z$ be positive real numbers that satisfy the following system of equations: \n\\[\\log_2\\left({x \\over yz}\\right) = {1 \\over 2}\\]\n\\[\\log_2\\left({y \\over xz}\\right) = {1 \\over 3}\\]\n\\[\\log_2\\left({z \\over xy}\\right) = {1 \\over 4}\\]\nThen the value of $\\left|\\log_2(x^4y^3z^2)\\right|$ is $\\tfrac{m}{n}$ where $m$ and $n$ are relatively prime positive integers. Find $m+n$.',
 'Solution': 'Denote $\\log_2(x) = a$, $\\log_2(y) = b$, and $\\log_2(z) = c$.\n\nThen, we have:\n$a-b-c = \\frac{1}{2}$,\n$-a+b-c = \\frac{1}{3}$,\n$-a-b+c = \\frac{1}{4}$.\n\nNow, we can solve to get $a = \\frac{-7}{24}, b = \\frac{-9}{24}, c = \\frac{-5}{12}$.\nPlugging these values in, we obtain $|4a + 3b + 2c|  = \\frac{25}{8} \\implies \\boxed{033}$.',
 'Answer': 33}

In [ ]:
len(aime_dataset['train'])

30

In [ ]:
aime_dataset['train'][0]['Problem']

'Let $x,y$ and $z$ be positive real numbers that satisfy the following system of equations: \n\\[\\log_2\\left({x \\over yz}\\right) = {1 \\over 2}\\]\n\\[\\log_2\\left({y \\over xz}\\right) = {1 \\over 3}\\]\n\\[\\log_2\\left({z \\over xy}\\right) = {1 \\over 4}\\]\nThen the value of $\\left|\\log_2(x^4y^3z^2)\\right|$ is $\\tfrac{m}{n}$ where $m$ and $n$ are relatively prime positive integers. Find $m+n$.'

In [ ]:
aime_dataset['train'][0]['Solution']

'Denote $\\log_2(x) = a$, $\\log_2(y) = b$, and $\\log_2(z) = c$.\n\nThen, we have:\n$a-b-c = \\frac{1}{2}$,\n$-a+b-c = \\frac{1}{3}$,\n$-a-b+c = \\frac{1}{4}$.\n\nNow, we can solve to get $a = \\frac{-7}{24}, b = \\frac{-9}{24}, c = \\frac{-5}{12}$.\nPlugging these values in, we obtain $|4a + 3b + 2c|  = \\frac{25}{8} \\implies \\boxed{033}$.'

In [ ]:
aime_dataset['train'][0]['Answer']

33

In [ ]:
# AIME 0
deepseek_inference_result = generate_response(system_message="",
                             user_message=aime_dataset['train'][0]['Problem'],
                                            tokenizer=deepseek_tokenizer,
                                            model=deepseek_model)
print(deepseek_inference_result)
# 정답 : 33

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


<think>
Okay, so I have this problem where I need to find the value of |log₂(x⁴y³z²)| given a system of logarithmic equations. Hmm, let me try to figure this out step by step.

First, let me write down the given equations:

1. log₂(x / (y z)) = 1/2
2. log₂(y / (x z)) = 1/3
3. log₂(z / (x y)) = 1/4

And I need to find |log₂(x⁴ y³ z²)|. That seems like it's asking for the absolute value of the logarithm base 2 of some expression involving x, y, z. Maybe I can express x, y, z in terms of each other or find their ratios?

Let me recall that log(a/b) = log a - log b. So, maybe I can rewrite each equation using this property.

Starting with the first equation:

log₂(x / (y z)) = log₂ x - log₂(y z) = log₂ x - (log₂ y + log₂ z) = 1/2.

Similarly, the second equation:

log₂(y / (x z)) = log₂ y - log₂(x z) = log₂ y - (log₂ x + log₂ z) = 1/3.

And the third equation:

log₂(z / (x y)) = log₂ z - log₂(x y) = log₂ z - (log₂ x + log₂ y) = 1/4.

Hmm, so if I let a = log₂ x, b = log₂ y, c = log₂ z, the

In [ ]:
aime_dataset['train'][1]['Problem']

'Let $O(0,0), A(\\tfrac{1}{2}, 0),$ and $B(0, \\tfrac{\\sqrt{3}}{2})$ be points in the coordinate plane. Let $\\mathcal{F}$ be the family of segments $\\overline{PQ}$ of unit length lying in the first quadrant with $P$ on the $x$-axis and $Q$ on the $y$-axis. There is a unique point $C$ on $\\overline{AB}$, distinct from $A$ and $B$, that does not belong to any segment from $\\mathcal{F}$ other than $\\overline{AB}$. Then $OC^2 = \\tfrac{p}{q}$, where $p$ and $q$ are relatively prime positive integers. Find $p + q$.'

In [ ]:
aime_dataset['train'][1]['Answer']

23

In [ ]:
# AIME 1
deepseek_inference_result = generate_response(system_message="",
                             user_message=aime_dataset['train'][1]['Problem'],
                                            tokenizer=deepseek_tokenizer,
                                            model=deepseek_model)
print(deepseek_inference_result)
# 정답 : 23

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


<think>
Okay, so I have this problem here with points O, A, and B given in the coordinate plane. O is at (0,0), A is at (1/2, 0), and B is at (0, sqrt(3)/2). Then there's this family of segments F, which are all the segments PQ of unit length lying in the first quadrant, with P on the x-axis and Q on the y-axis. The task is to find a unique point C on segment AB, different from A and B, that doesn't belong to any segment from F except for AB itself. Then, we need to compute OC squared, which is the square of the distance from the origin to point C, and express it as a reduced fraction p/q, then find p + q.

Alright, let's break this down step by step.

First, let me visualize the coordinate system. O is the origin, A is halfway along the x-axis from O to (1,0), and B is somewhere on the y-axis, specifically at (0, sqrt(3)/2). So, AB is a line segment connecting (1/2, 0) to (0, sqrt(3)/2). I can probably find the equation of line AB later on.

Next, the family F consists of all segments

In [ ]:
aime_dataset['train'][2]['Problem']

'Jen enters a lottery by picking $4$ distinct numbers from $S=\\{1,2,3,\\cdots,9,10\\}.$ $4$ numbers are randomly chosen from $S.$ She wins a prize if at least two of her numbers were $2$ of the randomly chosen numbers, and wins the grand prize if all four of her numbers were the randomly chosen numbers. The probability of her winning the grand prize given that she won a prize is $\\tfrac{m}{n}$ where $m$ and $n$ are relatively prime positive integers. Find $m+n$.'

In [ ]:
aime_dataset['train'][2]['Answer']

116

In [ ]:
# AIME 2
deepseek_inference_result = generate_response(system_message="",
                             user_message=aime_dataset['train'][2]['Problem'],
                                            tokenizer=deepseek_tokenizer,
                                            model=deepseek_model)
print(deepseek_inference_result)
# 정답 : 116

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


<think>
Okay, so I have this probability problem here, and I need to figure it out step by step. Let me read it again and make sure I understand.

Jen enters a lottery by picking 4 distinct numbers from the set S = {1,2,3,...,9,10}. So, S has numbers from 1 to 10, which is 10 numbers in total. She picks 4 distinct numbers from this set. Then, 4 numbers are randomly chosen from S. She wins a prize if at least two of her numbers are among the randomly chosen numbers. She wins the grand prize if all four of her numbers are the randomly chosen numbers. The question is asking for the probability of her winning the grand prize given that she won a prize. This probability is expressed as a fraction m/n, where m and n are coprime positive integers. We need to find m + n.

Alright, so let's break this down. First, the total number of possible lottery tickets Jen could have is the number of ways to choose 4 distinct numbers from 10, which is the combination C(10,4). Let me compute that.

C(10,4)

In [ ]:
aime_dataset['train'][3]['Problem']

"Alice and Bob play the following game. A stack of $n$ tokens lies before them. The players take turns with Alice going first. On each turn, the player removes either $1$ token or $4$ tokens from the stack. Whoever removes the last token wins. Find the number of positive integers $n$ less than or equal to $2024$ for which there exists a strategy for Bob that guarantees that Bob will win the game regardless of Alice's play."

In [ ]:
aime_dataset['train'][3]['Answer']

809

In [ ]:
# AIME 3
deepseek_inference_result = generate_response(system_message="",
                             user_message=aime_dataset['train'][3]['Problem'],
                                            tokenizer=deepseek_tokenizer,
                                            model=deepseek_model)
print(deepseek_inference_result)
# 정답 : 809

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


<think>
Okay, so Alice and Bob are playing this game where they take turns removing either 1 or 4 tokens from a stack. The person who takes the last token wins. I need to figure out how many positive integers n ≤ 2024 make it so that Bob can guarantee a win, no matter how Alice plays. Hmm, sounds like a combinatorial game theory problem. Maybe I can model this using winning and losing positions.

Let me recall, in such games, a position is a losing position if the player whose turn it is cannot force a win, assuming the opponent plays optimally. Conversely, a winning position is one where the current player can force a win. So, I need to figure out for each n, whether it's a winning or losing position.

Since the players can remove 1 or 4 tokens, the key is to see how these moves affect the number of tokens left. Let me try to list out the positions from 0 upwards and label them as winning (W) or losing (L).

Starting with n=0: If there are no tokens left, the game is over, so the play

In [ ]:
# 성능 분석
# DeepSeek-R1-Distill-Llama-8B : 4/4 - 100%

## 2. gpt-4o-2024-08-06 모델

In [ ]:
!pip install openai

In [ ]:
OPENAI_KEY = "여러분의_OPENAI_API_KEY"

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=OPENAI_KEY)

In [ ]:
response = client.chat.completions.create(
    model="gpt-4o-2024-08-06",
    messages=[
        {
            "role": "user",
            "content": aime_dataset['train'][0]['Problem']
        }
    ]
)
print(response.choices[0].message.content)
# 정답 : 33

Let's first rewrite the system of equations using the properties of logarithms:

\[
\log_2\left(\frac{x}{yz}\right) = \frac{1}{2} \\
\Rightarrow \log_2(x) - \log_2(y) - \log_2(z) = \frac{1}{2}
\]

\[
\log_2\left(\frac{y}{xz}\right) = \frac{1}{3} \\
\Rightarrow \log_2(y) - \log_2(x) - \log_2(z) = \frac{1}{3}
\]

\[
\log_2\left(\frac{z}{xy}\right) = \frac{1}{4} \\
\Rightarrow \log_2(z) - \log_2(x) - \log_2(y) = \frac{1}{4}
\]

Let \( a = \log_2(x) \), \( b = \log_2(y) \), \( c = \log_2(z) \). Then the equations become:

\[
a - b - c = \frac{1}{2}
\]
\[
b - a - c = \frac{1}{3}
\]
\[
c - a - b = \frac{1}{4}
\]

We want to find \( \left|\log_2(x^4y^3z^2)\right| = \left|4a + 3b + 2c\right| \).

First, add all three equations:

\[
(a - b - c) + (b - a - c) + (c - a - b) = \frac{1}{2} + \frac{1}{3} + \frac{1}{4}
\]

Simplifying the left side:

\[
a - b - c + b - a - c + c - a - b = -2a - 2b - 2c
\]

and solving for the right side:

\[
\frac{1}{2} + \frac{1}{3} + \frac{1}{4} = \frac{6 + 4 + 3}{

In [ ]:
response = client.chat.completions.create(
    model="gpt-4o-2024-08-06",
    messages=[
        {
            "role": "user",
            "content": aime_dataset['train'][1]['Problem']
        }
    ]
)

print(response.choices[0].message.content)
# 정답 : 23

To solve for the unique point \(C\) on \(\overline{AB}\), and find the squared distance \(OC^2\), we first parameterize \(\overline{PQ}\) from the family \(\mathcal{F}\) of unit length segments. Here \(P = (x, 0)\) is on the \(x\)-axis and \(Q = (0, y)\) is on the \(y\)-axis, with the condition that the length of \(\overline{PQ}\) is 1:

\[
\sqrt{x^2 + y^2} = 1
\]

Thus, 

\[
x^2 + y^2 = 1
\]

Next, we parameterize segment \(\overline{AB}\) with endpoints \(A\left(\frac{1}{2}, 0\right)\) and \(B\left(0, \frac{\sqrt{3}}{2}\right)\). A point \(C\) on \(\overline{AB}\) can be represented as:

\[
C = \left( \frac{1-t}{2}, \frac{t\sqrt{3}}{2} \right)
\]

for \(0 \le t \le 1\). We need to find the \(t\) such that \(C\) does not belong to any segment in \(\mathcal{F}\). \(C\) belongs to a segment \(\overline{PQ}\) if the coordinates of \(C\), \(\left(x = \frac{1-t}{2}, y = \frac{t\sqrt{3}}{2}\right)\), satisfy:

\[
x^2 + y^2 = 1
\]

Substituting \(x = \frac{1-t}{2}\) and \(y = \frac{t\sqrt{3}

In [ ]:
response = client.chat.completions.create(
    model="gpt-4o-2024-08-06",
    messages=[
        {
            "role": "user",
            "content": aime_dataset['train'][2]['Problem']
        }
    ]
)

print(response.choices[0].message.content)
# 정답 : 116

To solve the problem, let's first determine the total ways Jen can win a prize, and then the ways she can win the grand prize.

First, determine the total number of ways to choose 4 distinct numbers out of the set \(S = \{1, 2, 3, \ldots, 10\}\):

\[
\binom{10}{4} = 210
\]

Next, calculate the number of favorable outcomes where at least 2 of Jen’s chosen numbers are within the randomly chosen numbers. We will do this by considering the different valid cases and using Jen's potential ways to select her numbers.

**Case 1: Jen matches exactly 2 numbers.**

- Choose 2 matches from the 4 drawn: \(\binom{4}{2} = 6\)
- Choose 2 non-matching numbers from the remaining 6 available numbers: \(\binom{6}{2} = 15\)

The number of such favorable outcomes is:

\[
\binom{4}{2} \times \binom{6}{2} = 6 \times 15 = 90
\]

**Case 2: Jen matches exactly 3 numbers.**

- Choose 3 matches from the 4 drawn: \(\binom{4}{3} = 4\)
- Choose 1 non-matching number from the remaining 6 available numbers: \(\binom{6}

In [ ]:
response = client.chat.completions.create(
    model="gpt-4o-2024-08-06",
    messages=[
        {
            "role": "user",
            "content": aime_dataset['train'][3]['Problem']
        }
    ]
)

print(response.choices[0].message.content)
# 정답 : 809

To determine for how many positive integers \( n \) (where \( 1 \leq n \leq 2024 \)) Bob has a winning strategy, we need to analyze the game from the perspective of losing and winning positions.

A **losing position** is one where the player whose turn it is cannot win if the opponent plays optimally. Conversely, a **winning position** allows the current player to force a win with optimal play.

1. **Base Cases**:
   - If \( n = 0 \), Bob wins trivially as there are no tokens for Alice to play on her turn. This is a losing position for the first person to play.
   - If \( n = 1 \), Alice can immediately remove the single token and win.
   - If \( n = 2 \) or \( n = 3 \), Alice removes one token on her turn, leaving a situation similar to the \( n = 1 \) case, which she can win.
   - If \( n = 4 \), Alice can remove all four tokens, hence she can win directly.

2. **Finding Patterns:**
   - For \( n = 5 \), Alice can either remove 1 token (leaving 4, which is a winning position for her)

In [ ]:
# 성능 분석
# DeepSeek-R1-Distill-Llama-8B : 4/4 - 100%
# gpt-4o-2024-08-06 : 1/4 - 25%

## 3. Llama-3.1-8B-Instruct

In [ ]:
# 실행 전에 [런타임 - 세션 다시 시작]으로 세션 다시 시작하기
import os
os.environ['HF_TOKEN']="여러분의 HuggingFace Access Token값"

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

llama3_1_tokenizer = AutoTokenizer.from_pretrained(model_id)
llama3_1_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
# AIME 0
llama_3_1_inference_result = generate_response(system_message="",
                             user_message=aime_dataset['train'][0]['Problem'],
                                            tokenizer=llama3_1_tokenizer,
                                            model=llama3_1_model)
print(llama_3_1_inference_result)
# 정답 : 33

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexp

To solve this problem, we can start by using the properties of logarithms to rewrite the given system of equations.

We have that $\log_2\left({x \over yz}\right) = {1 \over 2}$, which can be rewritten as $\log_2(x) - \log_2(yz) = {1 \over 2}$.

Similarly, we have that $\log_2\left({y \over xz}\right) = {1 \over 3}$, which can be rewritten as $\log_2(y) - \log_2(xz) = {1 \over 3}$.

Finally, we have that $\log_2\left({z \over xy}\right) = {1 \over 4}$, which can be rewritten as $\log_2(z) - \log_2(xy) = {1 \over 4}$.

Now, we can add these three equations together to get:

$\log_2(x) + \log_2(y) + \log_2(z) - \log_2(xyz) = \frac{1}{2} + \frac{1}{3} + \frac{1}{4}$

Using the property of logarithms that $\log_a(b) + \log_a(c) = \log_a(bc)$, we can rewrite the left-hand side of the equation as:

$\log_2(xyz) - \log_2(xyz) = \frac{1}{2} + \frac{1}{3} + \frac{1}{4}$

This simplifies to $0 = \frac{1}{2} + \frac{1}{3} + \frac{1}{4}$, which is not true. However, we can rewrite the original sys

In [ ]:
# AIME 1
llama_3_1_inference_result = generate_response(system_message="",
                             user_message=aime_dataset['train'][1]['Problem'],
                                            tokenizer=llama3_1_tokenizer,
                                            model=llama3_1_model)
print(llama_3_1_inference_result)
# 정답 : 23

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


To find the point $C$ on $\overline{AB}$ that does not belong to any segment from $\mathcal{F}$ other than $\overline{AB}$, we need to find the point where the perpendicular bisector of $\overline{AB}$ intersects $\overline{AB}$.

First, we find the midpoint of $\overline{AB}$, which is $\left(\frac{1}{4}, \frac{\sqrt{3}}{4}\right)$. The slope of $\overline{AB}$ is $\frac{\frac{\sqrt{3}}{2}}{\frac{1}{2}} = \sqrt{3}$, so the slope of the perpendicular bisector is $\frac{-1}{\sqrt{3}}$.

The equation of the perpendicular bisector is $y - \frac{\sqrt{3}}{4} = \frac{-1}{\sqrt{3}}\left(x - \frac{1}{4}\right)$. We can simplify this to $y = \frac{-1}{\sqrt{3}}x + \frac{1}{4} + \frac{\sqrt{3}}{4}$.

Now, we need to find the intersection of this line with $\overline{AB}$. We can substitute $y = \frac{\sqrt{3}}{2}$ into the equation of the line to get $\frac{\sqrt{3}}{2} = \frac{-1}{\sqrt{3}}x + \frac{1}{4} + \frac{\sqrt{3}}{4}$. Solving for $x$, we get $x = \frac{1}{3}$.

Therefore, the point $

In [ ]:
# AIME 2
llama_3_1_inference_result = generate_response(system_message="",
                             user_message=aime_dataset['train'][2]['Problem'],
                                            tokenizer=llama3_1_tokenizer,
                                            model=llama3_1_model)
print(llama_3_1_inference_result)
# 정답 : 116

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


To find the probability of Jen winning the grand prize given that she won a prize, we need to find the probability of her winning the grand prize and the probability of her winning a prize, and then divide the former by the latter.

First, let's find the total number of ways to choose 4 numbers from the set S. This is given by the combination formula:

$\binom{10}{4} = \frac{10!}{4!6!} = 210$

Now, let's find the number of ways Jen can win a prize. This includes the cases where she wins the grand prize, and the cases where she wins a prize but not the grand prize.

The number of ways Jen can win the grand prize is 1, since there is only one way to choose all 4 numbers from the set S.

The number of ways Jen can win a prize but not the grand prize is the number of ways to choose 2 numbers from the set S that are not in Jen's 4 numbers, and 2 numbers from the set S that are in Jen's 4 numbers. This is given by:

$\binom{6}{2} \cdot \binom{4}{2} = 15 \cdot 6 = 90$

So, the total number of

In [ ]:
# AIME 3
llama_3_1_inference_result = generate_response(system_message="",
                             user_message=aime_dataset['train'][3]['Problem'],
                                            tokenizer=llama3_1_tokenizer,
                                            model=llama3_1_model)
print(llama_3_1_inference_result)
# 정답 : 809

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


To solve this problem, we need to find the number of positive integers $n$ less than or equal to $2024$ for which there exists a strategy for Bob that guarantees that Bob will win the game regardless of Alice's play.

We can use the following approach:

1.  Let's consider the possible remainders of $n$ modulo $5$. We can have $n \equiv 0, 1, 2, 3, 4 \pmod{5}$.
2.  If $n \equiv 0 \pmod{5}$, then Alice can always make a move that leaves a remainder of $1, 2, 3,$ or $4$ modulo $5$ for Bob. In this case, Bob can always make a move that leaves a remainder of $0$ modulo $5$ for Alice, and so on. This means that if $n \equiv 0 \pmod{5}$, then Bob will win the game.
3.  If $n \equiv 1 \pmod{5}$, then Alice can always make a move that leaves a remainder of $2, 3,$ or $4$ modulo $5$ for Bob. In this case, Bob can always make a move that leaves a remainder of $1$ modulo $5$ for Alice, and so on. This means that if $n \equiv 1 \pmod{5}$, then Alice will win the game.
4.  If $n \equiv 2 \pmod{5}$, 

|모델|0|1|2|3|
|------|---|---|---|---|
|DeepSeek-R1-Distill-Llama-8B|o|o|o|o|
|gpt-4o-2024-08-06|x|x|o|x|
|Llama-3.1-8B-Instruct|x|x|x|o|

In [ ]:
# 맞춘 갯수 분석
# 1위 : DeepSeek-R1-Distill-Llama-8B - 4/4
# 2위 : gpt-4o-2024-08-06,Llama-3.1-8B-Instruct - 1/4